# Notebook 09 — Frozen Cross-Encoder evaluation

Completed evaluation: 480 queries (240 lead, 240 late). This notebook restores frozen inputs and checkpoints from Google Drive, resumes missing predictions and exports metrics. No training or tuning.

## Reuse saved results
Copy the repository folder `results/reranking_eval_v1` to `/content/drive/MyDrive/greek-news-nlp/results/reranking_eval_v1`. Keep a single writer: do not run another notebook against the same checkpoint folder. Run cells in order. Existing verified checkpoints are reused; with all 480 present no model inference is needed. The pinned corpus is still downloaded if missing, to verify input fingerprints.

## Reproduction scope
The candidate pools and protocol are frozen inputs. The companion `09_cross_encoder_development.ipynb` records their preparation and development experiments; it is not needed to inspect final results. Do not rerun development to tune on these evaluation results.

The model revision is pinned. Package installation is not globally locked; the included version output records the final resume session only, not every earlier checkpoint session. CPU timings span resumed sessions and exclude model loading and Hybrid retrieval.

Maintenance after completion: refresh pending checkpoints before inference and validate an existing checkpoint before skipping it. Saved metrics and predictions are unchanged. Outputs below are retained from the completed run; they are not a rerun of this maintenance edit.


In [7]:
# NOTEBOOK 09 — FINAL ALL-IN-ONE CELL
# Colab CPU: restore corpus -> resume checkpointed inference -> final metrics & report.
# After a Colab reset, run all cells in order. No training, no tuning.

import sys, subprocess, json, hashlib, time, os, importlib.util
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

# ============================ SETTINGS ============================
MAX_NEW_QUERIES = 480     # Run all remaining queries automatically
BASE = Path('/content/drive/MyDrive/greek-news-nlp')
OUT = BASE / 'results/reranking_eval_v1/frozen_eval_v1'
PARENT = OUT.parent
CKPT = OUT / 'cross_encoder_checkpoints'
MODEL_ID = 'BAAI/bge-reranker-v2-m3'
MODEL_REV = '953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e'
DATASET_REV = '17c079f7c9c6a71a8a1747d8a219594260096d0a'
EXPECTED_DOCS_SHA = '8ca6a64c934769dfd09e09635013fa73cf7f80dfa70c4cc045f67958ffe51c93'
EXPECTED_CAND_SHA = '40ca7f1af52e76d0be1bb739b4ae64600561b969470f2a9fe151f4659d7c7581'
EXPECTED_QUERY_SHA = '2a5c54e14313fb1a3f5dae51192a45cdce748dadafd58093d6e46fc0e870fecc'
TOP_K = 5
MAX_LENGTH = 512
BATCH_SIZE = 2
DEVICE = 'cpu'

# ============================ HELPERS ============================
def sha_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha_file(path):
    return sha_bytes(Path(path).read_bytes())

def json_bytes(item):
    return (json.dumps(item, ensure_ascii=False, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def save_once(path, content):
    path = Path(path)
    if path.exists():
        if path.read_bytes() != content:
            raise RuntimeError(f'Existing artifact differs; refusing overwrite: {path}')
        return
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_bytes(content)
    os.replace(tmp, path)
    assert path.read_bytes() == content

# ============================ 1. MOUNT DRIVE ============================
try:
    from google.colab import drive
except ImportError as ex:
    raise RuntimeError('Run this code in Google Colab, not locally.') from ex
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert BASE.is_dir(), f'Google Drive folder missing: {BASE}'
assert OUT.is_dir(), f'Run original Cell 28 first: {OUT}'
CKPT.mkdir(parents=True, exist_ok=True)
print('NOTEBOOK 09 — FINAL AUTOMATIC RESUME + EVALUATION', flush=True)

# ============================ 2. VERIFY ALL FROZEN INPUTS ============================
protocol_file = PARENT / 'reranking_protocol_v1.json'
setup_file = OUT / 'evaluation_setup.json'
query_file = OUT / 'evaluation_queries_frozen.jsonl'
candidates_file = OUT / 'evaluation_hybrid_candidates.jsonl'
manifest_file = OUT / 'hybrid_candidates_manifest.json'
dev_report_file = PARENT / 'dev_top5_report_60.json'
for path in [protocol_file, setup_file, query_file, candidates_file, manifest_file, dev_report_file]:
    assert path.is_file(), f'Missing frozen input: {path}'

protocol = read_json(protocol_file)
setup = read_json(setup_file)
candidates_manifest = read_json(manifest_file)
dev_report = read_json(dev_report_file)
protocol_sha = sha_file(protocol_file)
setup_sha = sha_file(setup_file)
queries_sha = sha_file(query_file)
candidates_sha = sha_file(candidates_file)
assert candidates_sha == EXPECTED_CAND_SHA
assert queries_sha == EXPECTED_QUERY_SHA
assert setup['protocol_sha256'] == protocol_sha
assert setup['frozen_queries_sha256'] == queries_sha
assert candidates_manifest['status'] == 'complete'
assert candidates_manifest['candidate_file_sha256'] == candidates_sha
assert candidates_manifest['frozen_queries_sha256'] == queries_sha
assert candidates_manifest['protocol_sha256'] == protocol_sha
assert candidates_manifest['evaluation_setup_sha256'] == setup_sha
assert dev_report['status'] == 'development_complete'
assert dev_report['n_queries'] == 60
assert dev_report['protocol_sha256'] == protocol_sha
assert dev_report['evaluation_queries_used'] is False
assert protocol['model']['id'] == MODEL_ID
assert protocol['model']['revision'] == MODEL_REV
assert protocol['model']['max_length'] == MAX_LENGTH
assert protocol['model']['batch_size'] == BATCH_SIZE
assert protocol['model']['device'] == DEVICE
assert protocol['retrieval']['rerank_top_k'] == TOP_K
assert protocol['retrieval']['bm25_candidate_k'] == 50
assert protocol['retrieval']['dense_candidate_k'] == 50
assert protocol['retrieval']['rrf_k'] == 60

queries = read_jsonl(query_file)
pools = read_jsonl(candidates_file)
assert len(queries) == len(pools) == 480
assert Counter(q['query_type'] for q in queries) == {'lead': 240, 'late': 240}
assert len({q['query_id'] for q in queries}) == 480
for q, pool in zip(queries, pools):
    assert all(q[k] == pool[k] for k in ['query_id', 'query_type', 'query', 'source_id'])
    assert q['partition'] in ('eval', 'evaluation')
    assert pool['partition'] == 'evaluation'
    candidates = pool['candidates']
    assert 5 <= len(candidates) <= 100
    assert [c['rank'] for c in candidates] == list(range(1, len(candidates) + 1))
    ids = [c['document_id'] for c in candidates]
    assert len(ids) == len(set(ids))
print('Frozen protocol, queries, candidate pools: VERIFIED', flush=True)

# ============================ 3. RESTORE EXACT DOCUMENT TEXT ============================
docs_path = BASE / 'rag_index/docs.json'
if not docs_path.exists():
    print('Drive docs.json missing; downloading exact pinned index document list...', flush=True)
    if importlib.util.find_spec('huggingface_hub') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])
    from huggingface_hub import hf_hub_download
    docs_path = Path(hf_hub_download(
        repo_id='Tilemachos-P/greek-news-index', repo_type='dataset',
        filename='docs.json', revision=DATASET_REV
    ))
assert sha_file(docs_path) == EXPECTED_DOCS_SHA, 'Corpus SHA mismatch — refusing inference.'
docs_list = read_json(docs_path)
assert isinstance(docs_list, list) and len(docs_list) == 5250
assert all(isinstance(d, dict) and isinstance(d.get('id'), str) and isinstance(d.get('text'), str) for d in docs_list)
doc_by_id = {d['id']: d for d in docs_list}
assert len(doc_by_id) == 5250
assert all(c['document_id'] in doc_by_id for p in pools for c in p['candidates'][:TOP_K])
print('Pinned full-text corpus: VERIFIED (5,250 docs)', flush=True)

# ============================ 4. VALIDATE SAVED CHECKPOINTS ============================
def ckpt_path(pool):
    return CKPT / f"{pool['query_id']}.json"

def prepare(pool):
    top5 = pool['candidates'][:TOP_K]
    input_fingerprint = sha_bytes(json_bytes({
        'query': pool['query'],
        'documents': [
            {'document_id': c['document_id'], 'text_sha256': sha_bytes(doc_by_id[c['document_id']]['text'].encode('utf-8'))}
            for c in top5
        ],
    }))
    return top5, input_fingerprint

def verify_checkpoint(pool, saved):
    top5, input_fingerprint = prepare(pool)
    original_ids = [c['document_id'] for c in top5]
    assert saved['query_id'] == pool['query_id']
    assert saved['query_type'] == pool['query_type']
    assert saved['partition'] == 'evaluation'
    assert saved['model_id'] == MODEL_ID
    assert saved['model_revision'] == MODEL_REV
    assert saved['protocol_sha256'] == protocol_sha
    assert saved['candidate_sha256'] == candidates_sha
    assert saved['input_sha256'] == input_fingerprint
    assert saved['max_length'] == MAX_LENGTH
    assert saved['batch_size'] == BATCH_SIZE
    assert saved['rerank_top_k'] == TOP_K
    assert saved['hybrid_top5_ids'] == original_ids
    ranked = saved['reranked_top5']
    assert len(ranked) == 5
    assert len({r['document_id'] for r in ranked}) == 5
    assert set(r['document_id'] for r in ranked) == set(original_ids)
    for item in ranked:
        assert np.isfinite(item['reranker_score'])
        assert item['hybrid_rank'] == original_ids.index(item['document_id']) + 1
    assert ranked == sorted(ranked, key=lambda r: (-r['reranker_score'], r['hybrid_rank'], r['document_id']))
    assert np.isfinite(saved['inference_seconds']) and saved['inference_seconds'] >= 0

pending = []
for pool in pools:
    path = ckpt_path(pool)
    if path.exists():
        verify_checkpoint(pool, read_json(path))
    else:
        pending.append(pool)
print(f'Existing checkpoints: {480 - len(pending)}/480 — VERIFIED', flush=True)
print(f'Queries remaining: {len(pending)}', flush=True)


NOTEBOOK 09 — FINAL AUTOMATIC RESUME + EVALUATION
Frozen protocol, queries, candidate pools: VERIFIED
Pinned full-text corpus: VERIFIED (5,250 docs)
Existing checkpoints: 320/480 — VERIFIED
Queries remaining: 160


In [8]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version)
for package in ("torch", "transformers", "sentence-transformers", "numpy", "pandas"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
torch: 2.11.0+cpu
transformers: 5.18.0
sentence-transformers: 5.7.0
numpy: 2.1.3
pandas: 2.2.3


In [9]:
# Refresh pending work when this cell is rerun in the same runtime.
pending = []
for pool in pools:
    path = ckpt_path(pool)
    if path.exists():
        verify_checkpoint(pool, read_json(path))
    else:
        pending.append(pool)
print(f"Verified checkpoints: {len(pools) - len(pending)}/{len(pools)}", flush=True)

# ============================ 5. LOAD MODEL ONLY IF NEEDED ============================
if pending:
    if importlib.util.find_spec('sentence_transformers') is None:
        print('Installing Cross-Encoder dependency...', flush=True)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'])
    from sentence_transformers import CrossEncoder
    print(f'Loading {MODEL_ID} on CPU (pinned revision)...', flush=True)
    reranker = CrossEncoder(
        MODEL_ID, revision=MODEL_REV, device=DEVICE,
        max_length=MAX_LENGTH, trust_remote_code=False,
    )
    print('Cross-Encoder ready.', flush=True)
    print(f'Processing up to {min(MAX_NEW_QUERIES, len(pending))} remaining queries...', flush=True)
    started = time.perf_counter()
    for n, pool in enumerate(pending[:MAX_NEW_QUERIES], start=1):
        if ckpt_path(pool).exists():
            verify_checkpoint(pool, read_json(ckpt_path(pool)))
            continue
        top5, fingerprint = prepare(pool)
        pairs = [(pool['query'], doc_by_id[c['document_id']]['text']) for c in top5]
        t0 = time.perf_counter()
        scores = np.asarray(reranker.predict(
            pairs, batch_size=BATCH_SIZE, show_progress_bar=False, convert_to_numpy=True
        ), dtype=float).reshape(-1)
        seconds = time.perf_counter() - t0
        assert scores.shape == (TOP_K,) and np.isfinite(scores).all()
        ranked = [
            {'document_id': c['document_id'], 'hybrid_rank': int(c['rank']), 'reranker_score': float(score)}
            for c, score in zip(top5, scores)
        ]
        ranked.sort(key=lambda r: (-r['reranker_score'], r['hybrid_rank'], r['document_id']))
        record = {
            'query_id': pool['query_id'], 'query_type': pool['query_type'],
            'partition': 'evaluation', 'model_id': MODEL_ID,
            'model_revision': MODEL_REV, 'protocol_sha256': protocol_sha,
            'candidate_sha256': candidates_sha, 'input_sha256': fingerprint,
            'rerank_top_k': TOP_K, 'max_length': MAX_LENGTH, 'batch_size': BATCH_SIZE,
            'hybrid_top5_ids': [c['document_id'] for c in top5],
            'reranked_top5': ranked, 'inference_seconds': float(seconds),
        }
        verify_checkpoint(pool, record)
        save_once(ckpt_path(pool), json_bytes(record))
        print(f'[{n}/{min(MAX_NEW_QUERIES, len(pending))}] {pool["query_id"]}: {seconds:.1f}s | SAVED', flush=True)
    print(f'Current run: {(time.perf_counter()-started)/60:.1f} minutes', flush=True)

remaining = [p for p in pools if not ckpt_path(p).exists()]
print(f'FINAL INFERENCE PROGRESS: {480-len(remaining)}/480', flush=True)
if remaining:
    print('Results are safely checkpointed. Re-run THIS SAME CELL to finish remaining queries.', flush=True)
else:
    print('All 480 queries completed. Generating FINAL evaluation metrics...', flush=True)

    # ====================== 6. FINAL PREDICTION ARTIFACTS ======================
    records = []
    for p in pools:
        record = read_json(ckpt_path(p))
        verify_checkpoint(p, record)
        records.append(record)
    predictions_bytes = b''.join(json_bytes(r) for r in records)
    predictions_file = OUT / 'cross_encoder_predictions_480.jsonl'
    save_once(predictions_file, predictions_bytes)
    predictions_sha = sha_bytes(predictions_bytes)
    prediction_manifest = {
        'experiment': 'cross_encoder_frozen_eval_v1', 'status': 'inference_complete',
        'n_queries': 480, 'model': protocol['model'], 'retrieval': protocol['retrieval'],
        'protocol_sha256': protocol_sha, 'evaluation_setup_sha256': setup_sha,
        'frozen_queries_sha256': queries_sha, 'candidate_sha256': candidates_sha,
        'predictions_file': predictions_file.name, 'predictions_sha256': predictions_sha,
        'evaluation_metrics_computed': False,
    }
    save_once(OUT / 'cross_encoder_predictions_manifest.json', json_bytes(prediction_manifest))

    # ====================== 7. RANKS & METRICS ======================
    rows = []
    for pool, record in zip(pools, records):
        ids = [c['document_id'] for c in pool['candidates']]
        ranked_top5 = [r['document_id'] for r in record['reranked_top5']]
        reranked_ids = ranked_top5 + ids[TOP_K:]
        gold = pool['source_id']
        base_rank = ids.index(gold)+1 if gold in ids else None
        new_rank = reranked_ids.index(gold)+1 if gold in reranked_ids else None
        def rank_key(rank): return float(rank) if rank is not None else float('inf')
        outcome = ('IMPROVED' if rank_key(new_rank) < rank_key(base_rank)
                   else 'DEGRADED' if rank_key(new_rank) > rank_key(base_rank)
                   else 'UNCHANGED')
        rows.append({
            'query_id': pool['query_id'], 'query_type': pool['query_type'],
            'source_id': gold, 'hybrid_gold_rank': base_rank,
            'reranker_gold_rank': new_rank, 'outcome': outcome,
            'inference_seconds': record['inference_seconds'],
        })
    final_df = pd.DataFrame(rows)
    assert len(final_df) == 480 and final_df.query_id.is_unique

    def compute_metrics(ranks):
        vals = pd.to_numeric(ranks, errors='coerce').to_numpy(dtype=float)
        rr = np.where((vals >= 1) & (vals <= 10), 1 / vals, 0.0)
        return {
            'recall@1': float(np.mean(vals == 1)),
            'recall@5': float(np.mean(vals <= 5)),
            'recall@10': float(np.mean(vals <= 10)),
            'mrr@10': float(np.mean(rr)),
        }

    metric_rows = []
    aggregate_metrics = {}
    for split, frame in [('lead', final_df[final_df.query_type=='lead']),
                         ('late', final_df[final_df.query_type=='late']),
                         ('overall', final_df)]:
        assert len(frame) == (480 if split == 'overall' else 240)
        hybrid = compute_metrics(frame.hybrid_gold_rank)
        cross = compute_metrics(frame.reranker_gold_rank)
        aggregate_metrics[split] = {'hybrid_rrf': hybrid, 'cross_encoder': cross}
        for metric_name in hybrid:
            metric_rows.append({
                'query_subset': split, 'metric': metric_name,
                'hybrid_rrf': hybrid[metric_name],
                'cross_encoder': cross[metric_name],
                'difference': cross[metric_name]-hybrid[metric_name],
            })
        # Frozen Top-5 reranking cannot change recall@5 / @10.
        assert np.isclose(hybrid['recall@5'], cross['recall@5'])
        assert np.isclose(hybrid['recall@10'], cross['recall@10'])

    metrics_df = pd.DataFrame(metric_rows)
    print('\nFINAL FROZEN EVALUATION — 480 QUERIES')
    print(metrics_df.to_string(index=False, float_format=lambda n: f'{n:.4f}'))
    print('\nRANKING OUTCOMES BY TYPE')
    print(pd.crosstab(final_df.query_type, final_df.outcome).to_string())

    predictions_csv_bytes = final_df.to_csv(index=False, lineterminator='\n', float_format='%.8f').encode('utf-8')
    metrics_csv_bytes = metrics_df.to_csv(index=False, lineterminator='\n', float_format='%.8f').encode('utf-8')
    detail_path = OUT / 'evaluation_query_results_480.csv'
    metric_path = OUT / 'evaluation_metrics_480.csv'
    save_once(detail_path, predictions_csv_bytes)
    save_once(metric_path, metrics_csv_bytes)

    report = {
        'experiment': 'cross_encoder_frozen_evaluation_v1', 'status': 'complete',
        'n_queries': 480, 'n_lead': 240, 'n_late': 240,
        'protocol_sha256': protocol_sha, 'development_report_sha256': sha_file(dev_report_file),
        'frozen_queries_sha256': queries_sha, 'candidate_sha256': candidates_sha,
        'corpus_docs_sha256': EXPECTED_DOCS_SHA,
        'model': protocol['model'], 'retrieval': protocol['retrieval'],
        'metrics': aggregate_metrics, 'ranking_outcomes': dict(Counter(final_df.outcome)),
        'total_measured_inference_seconds': float(final_df.inference_seconds.sum()),
        'mean_inference_seconds_per_query': float(final_df.inference_seconds.mean()),
        'artifacts': {
            'cross_encoder_predictions_480.jsonl': predictions_sha,
            detail_path.name: sha_bytes(predictions_csv_bytes),
            metric_path.name: sha_bytes(metrics_csv_bytes),
        },
        'limitations': protocol['limitations'],
        'notes': [
            'Synthetic query excerpts from source documents; gold is exact source-document identity, not independent human relevance.',
            'Top-5 reranking only; positions after rank 5 retain Hybrid RRF order.',
            'Cross-Encoder input pairs are truncated at 512 tokens.',
            'Evaluation labels were not used to train or tune the model.',
            'Inference time excludes model loading and Hybrid retrieval; CPU timings are not API latency.',
            'The deployed API still uses Dense E5; Hybrid and Cross-Encoder are offline experiments.',
        ],
    }
    report_path = OUT / 'evaluation_final_report_480.json'
    save_once(report_path, json_bytes(report))
    print('\nFINAL ARTIFACTS VERIFIED')
    for path in [predictions_file, OUT/'cross_encoder_predictions_manifest.json', detail_path, metric_path, report_path]:
        print(' -', path)
    print('\nOK — NOTEBOOK 09 EVALUATION COMPLETE (480/480)', flush=True)


Loading BAAI/bge-reranker-v2-m3 on CPU (pinned revision)...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Cross-Encoder ready.
Processing up to 160 remaining queries...
[1/160] late_141: 20.9s | SAVED
[2/160] late_142: 25.4s | SAVED
[3/160] late_143: 19.7s | SAVED
[4/160] late_144: 21.7s | SAVED
[5/160] late_145: 19.3s | SAVED
[6/160] late_146: 21.2s | SAVED
[7/160] late_147: 16.6s | SAVED
[8/160] late_148: 19.9s | SAVED
[9/160] late_149: 21.4s | SAVED
[10/160] late_150: 21.5s | SAVED
[11/160] late_151: 19.8s | SAVED
[12/160] late_152: 18.6s | SAVED
[13/160] late_153: 20.5s | SAVED
[14/160] late_154: 21.6s | SAVED
[15/160] late_155: 19.4s | SAVED
[16/160] late_156: 20.0s | SAVED
[17/160] late_157: 21.5s | SAVED
[18/160] late_158: 21.2s | SAVED
[19/160] late_159: 20.7s | SAVED
[20/160] late_160: 21.3s | SAVED
[21/160] late_161: 21.1s | SAVED
[22/160] late_162: 15.5s | SAVED
[23/160] late_163: 18.6s | SAVED
[24/160] late_164: 19.9s | SAVED
[25/160] late_165: 19.5s | SAVED
[26/160] late_166: 19.9s | SAVED
[27/160] late_167: 19.4s | SAVED
[28/160] late_168: 21.2s | SAVED
[29/160] late_169: 20.